# EP15. 에이전트 토큰 경제학

## 학습 목표

1. **에이전트 Multi-turn Loop**에서 토큰 비용이 O(n²)으로 증가하는 구조를 이해한다
2. **입력/출력 토큰 비대칭 가격** 구조를 분석하고 모델별 비용을 비교한다
3. **Langfuse 비용 태깅 및 대시보드**를 구축하여 실시간 비용을 추적한다
4. **임계값 기반 알림 시스템**과 **비용 최적화 전략**(Model Routing, Compression, Caching)을 설계한다

## AI 가이드

> 막히는 부분은 Claude에게 물어보세요.
> 예시 프롬프트:
> - "에이전트의 Multi-turn Loop에서 토큰이 왜 O(n²)으로 증가하는지 설명해줘"
> - "Langfuse CallbackHandler로 비용 태깅하는 방법을 알려줘"

**사전 요구사항**: Python 기본 문법, LLM API 호출 경험

**예상 소요 시간**: 60분

**필요한 API 키**:
- `ANTHROPIC_API_KEY`
- `OPENAI_API_KEY` (선택)
- `LANGFUSE_PUBLIC_KEY` / `LANGFUSE_SECRET_KEY`

---
## 섹션 1. 환경 설정

In [ ]:
# 필요한 패키지 설치
!uv pip install langchain langchain-anthropic langchain-openai langfuse tiktoken matplotlib pandas python-dotenv --quiet

---
## 섹션 2. 라이브러리 로드 + API 키 설정

In [ ]:
import os
import json
import time
from datetime import datetime, timedelta
from dataclasses import dataclass
from enum import Enum

import tiktoken
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

from dotenv import load_dotenv
load_dotenv()

# matplotlib 한글 폰트 설정 (macOS)
matplotlib.rcParams['font.family'] = 'AppleGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

print("라이브러리 로드 완료 ✓")

In [ ]:
# API 키 설정
# .env 파일 또는 환경 변수에서 로드됩니다
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "your-anthropic-api-key")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "your-openai-api-key")
LANGFUSE_PUBLIC_KEY = os.getenv("LANGFUSE_PUBLIC_KEY", "your-langfuse-public-key")
LANGFUSE_SECRET_KEY = os.getenv("LANGFUSE_SECRET_KEY", "your-langfuse-secret-key")

# Anthropic API 키 검증
assert ANTHROPIC_API_KEY != "your-anthropic-api-key", \
    "ANTHROPIC_API_KEY를 설정해주세요. (.env 파일 또는 환경 변수)"

os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["LANGFUSE_PUBLIC_KEY"] = LANGFUSE_PUBLIC_KEY
os.environ["LANGFUSE_SECRET_KEY"] = LANGFUSE_SECRET_KEY

print("API 키 설정 완료 ✓")

---
## 섹션 3. 단일 대화 vs 에이전트 비용 시뮬레이션

챗봇(1회 호출)과 에이전트(Multi-turn Loop)의 비용 차이를 직접 계산해봅니다.

In [ ]:
# tiktoken으로 토큰 수를 카운트하는 유틸리티 함수
# (Claude 토큰과 정확히 동일하지 않지만 근사치로 활용)
enc = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    """텍스트의 토큰 수를 반환합니다."""
    return len(enc.encode(text))

def calc_cost(input_tokens: int, output_tokens: int,
              input_price: float = 3.0, output_price: float = 15.0) -> float:
    """비용을 계산합니다. (가격 단위: $/M tokens)"""
    return (input_tokens * input_price + output_tokens * output_price) / 1_000_000

# 테스트
sample_text = "에이전트 토큰 경제학을 이해하는 것은 LLM 운영에서 매우 중요합니다."
print(f"샘플 텍스트 토큰 수: {count_tokens(sample_text)}")
print(f"비용 계산 예시 (1000 입력 + 500 출력, Sonnet 4): ${calc_cost(1000, 500):.6f}")
print("완료 ✓")

In [ ]:
# === 챗봇(1회 호출) vs 에이전트(Multi-turn) 비용 비교 ===

# 시나리오: "이번 분기 매출 보고서를 분석해줘"
system_prompt = """당신은 금융 분석 전문가입니다. 
데이터를 분석하고 인사이트를 제공합니다.
한국어로 답변해주세요."""

user_query = "이번 분기 매출 보고서를 분석하고, 핵심 트렌드와 개선 방안을 제시해줘."

# 1) 챗봇: 단일 호출
chatbot_input_tokens = count_tokens(system_prompt) + count_tokens(user_query)
chatbot_output_tokens = 500  # 예상 출력
chatbot_cost = calc_cost(chatbot_input_tokens, chatbot_output_tokens)

# 2) 에이전트: 5턴 루프 시뮬레이션
agent_turns = [
    {"name": "Turn 1: 계획 수립", "new_output": 300},
    {"name": "Turn 2: 데이터 조회 도구 호출", "new_output": 800},
    {"name": "Turn 3: 트렌드 분석", "new_output": 1200},
    {"name": "Turn 4: 개선 방안 도출", "new_output": 1500},
    {"name": "Turn 5: 최종 보고서 작성", "new_output": 2000},
]

base_input = count_tokens(system_prompt) + count_tokens(user_query)
accumulated_context = base_input
total_agent_input = 0
total_agent_output = 0

print("=" * 65)
print(f"{'턴':<30} {'입력 토큰':>10} {'출력 토큰':>10} {'누적 비용':>10}")
print("=" * 65)

running_cost = 0.0
for turn in agent_turns:
    input_tokens = accumulated_context
    output_tokens = turn["new_output"]
    turn_cost = calc_cost(input_tokens, output_tokens)
    running_cost += turn_cost
    
    total_agent_input += input_tokens
    total_agent_output += output_tokens
    accumulated_context += output_tokens  # 컨텍스트 누적!
    
    print(f"{turn['name']:<30} {input_tokens:>10,} {output_tokens:>10,} ${running_cost:>9.4f}")

agent_cost = calc_cost(total_agent_input, total_agent_output)

print("=" * 65)
print(f"\n📊 비용 비교 결과")
print(f"  챗봇 (1회):  입력 {chatbot_input_tokens:,} + 출력 {chatbot_output_tokens:,} = ${chatbot_cost:.4f}")
print(f"  에이전트 (5턴): 입력 {total_agent_input:,} + 출력 {total_agent_output:,} = ${agent_cost:.4f}")
print(f"  비용 배수:   {agent_cost / chatbot_cost:.1f}x")
print("완료 ✓")

---
## 섹션 4. Multi-turn Loop 비용 시뮬레이터

턴 수가 증가할수록 입력 토큰이 어떻게 누적되는지 시뮬레이션합니다.
10턴 에이전트에서 각 턴의 입력/출력 토큰과 비용을 추적합니다.

In [ ]:
import random

def simulate_agent_turns(n_turns: int, base_tokens: int = 1000,
                          avg_output: int = 800, output_std: int = 200,
                          input_price: float = 3.0, output_price: float = 15.0):
    """
    Multi-turn 에이전트의 토큰 비용을 시뮬레이션합니다.
    
    Args:
        n_turns: 총 턴 수
        base_tokens: 시스템 프롬프트 + 초기 질문 토큰
        avg_output: 턴당 평균 출력 토큰
        output_std: 출력 토큰 표준편차
        input_price: 입력 토큰 가격 ($/M)
        output_price: 출력 토큰 가격 ($/M)
    
    Returns:
        DataFrame with per-turn metrics
    """
    random.seed(42)  # 재현 가능성
    
    records = []
    accumulated = base_tokens
    total_cost = 0.0
    
    for turn in range(1, n_turns + 1):
        input_tokens = accumulated
        output_tokens = max(200, int(random.gauss(avg_output, output_std)))
        
        turn_cost = calc_cost(input_tokens, output_tokens, input_price, output_price)
        total_cost += turn_cost
        
        records.append({
            "턴": turn,
            "입력 토큰": input_tokens,
            "출력 토큰": output_tokens,
            "턴 비용 ($)": round(turn_cost, 6),
            "누적 비용 ($)": round(total_cost, 6),
        })
        
        # 다음 턴: 이전 출력이 컨텍스트에 추가됨
        accumulated += output_tokens
    
    return pd.DataFrame(records)

# 10턴 시뮬레이션 실행 (Claude Sonnet 4 기준)
df_sim = simulate_agent_turns(n_turns=10, base_tokens=1000,
                               avg_output=800, output_std=200,
                               input_price=3.0, output_price=15.0)

print("📊 10턴 에이전트 비용 시뮬레이션 (Claude Sonnet 4)")
print(df_sim.to_string(index=False))
print(f"\n총 입력 토큰: {df_sim['입력 토큰'].sum():,}")
print(f"총 출력 토큰: {df_sim['출력 토큰'].sum():,}")
print(f"총 비용: ${df_sim['누적 비용 ($)'].iloc[-1]:.4f}")
print("완료 ✓")

In [ ]:
# 턴별 입력 토큰 및 비용 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1) 턴별 입력 토큰 (누적 증가)
axes[0].bar(df_sim["턴"], df_sim["입력 토큰"], color="#42A5F5", alpha=0.8, label="입력 토큰")
axes[0].bar(df_sim["턴"], df_sim["출력 토큰"], bottom=df_sim["입력 토큰"],
            color="#EF5350", alpha=0.8, label="출력 토큰")
axes[0].set_xlabel("턴")
axes[0].set_ylabel("토큰 수")
axes[0].set_title("턴별 입력/출력 토큰 (컨텍스트 누적)")
axes[0].legend()
axes[0].set_xticks(range(1, 11))

# 2) 누적 비용 곡선
axes[1].plot(df_sim["턴"], df_sim["누적 비용 ($)"], marker="o", linewidth=2,
             color="#FF7043", markersize=8)
axes[1].fill_between(df_sim["턴"], df_sim["누적 비용 ($)"], alpha=0.2, color="#FF7043")
axes[1].set_xlabel("턴")
axes[1].set_ylabel("누적 비용 ($)")
axes[1].set_title("턴별 누적 비용 (O(n²) 성장)")
axes[1].set_xticks(range(1, 11))

plt.tight_layout()
plt.show()
print("완료 ✓")

---
## 섹션 5. 입력/출력 토큰 비대칭 실험

같은 질문을 보내도 **출력 토큰이 입력보다 4~8배 비싸기 때문에**,
토큰 수가 적더라도 출력 비용이 전체의 60% 이상을 차지합니다.

In [ ]:
# 입력/출력 비대칭 비용 분석
# 실제 API 호출 없이 시뮬레이션으로 비교합니다

# 시나리오: 에이전트 10턴 완료 후 총 토큰 분포
total_input = df_sim["입력 토큰"].sum()   # 시뮬레이션 결과 사용
total_output = df_sim["출력 토큰"].sum()

# 모델별 비용 계산
models = {
    "Claude Sonnet 4": {"input": 3.0, "output": 15.0},
    "Claude Haiku 3.5": {"input": 0.8, "output": 4.0},
    "GPT-4o": {"input": 2.5, "output": 10.0},
    "GPT-4o mini": {"input": 0.15, "output": 0.6},
    "Gemini 2.5 Pro": {"input": 1.25, "output": 10.0},
    "Gemini 2.5 Flash": {"input": 0.15, "output": 0.6},
}

results = []
for model_name, prices in models.items():
    input_cost = total_input * prices["input"] / 1_000_000
    output_cost = total_output * prices["output"] / 1_000_000
    total_cost = input_cost + output_cost
    output_ratio = output_cost / total_cost * 100 if total_cost > 0 else 0
    
    results.append({
        "모델": model_name,
        "입력 비용 ($)": round(input_cost, 4),
        "출력 비용 ($)": round(output_cost, 4),
        "총 비용 ($)": round(total_cost, 4),
        "출력 비율 (%)": round(output_ratio, 1),
    })

df_asym = pd.DataFrame(results)
print(f"📊 입력/출력 비대칭 분석 (입력 {total_input:,} + 출력 {total_output:,} 토큰)")
print(df_asym.to_string(index=False))
print("\n💡 모든 모델에서 출력 비용이 전체의 50% 이상을 차지합니다!")
print("완료 ✓")

In [ ]:
# 입력/출력 비용 비대칭 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1) 모델별 입력 vs 출력 비용 스택 바 차트
x = range(len(df_asym))
axes[0].barh(df_asym["모델"], df_asym["입력 비용 ($)"], color="#42A5F5", label="입력 비용")
axes[0].barh(df_asym["모델"], df_asym["출력 비용 ($)"], left=df_asym["입력 비용 ($)"],
             color="#EF5350", label="출력 비용")
axes[0].set_xlabel("비용 ($)")
axes[0].set_title("모델별 입력 vs 출력 비용")
axes[0].legend()

# 2) 출력 비용 비율 파이 차트 (Claude Sonnet 4 기준)
sonnet_row = df_asym[df_asym["모델"] == "Claude Sonnet 4"].iloc[0]
axes[1].pie(
    [sonnet_row["입력 비용 ($)"], sonnet_row["출력 비용 ($)"]],
    labels=["입력 토큰 비용", "출력 토큰 비용"],
    colors=["#42A5F5", "#EF5350"],
    autopct="%1.1f%%",
    startangle=90,
    textprops={"fontsize": 12}
)
axes[1].set_title("Claude Sonnet 4 비용 구성")

plt.tight_layout()
plt.show()
print("완료 ✓")

---
## 섹션 6. Context Accumulation O(n²) 시뮬레이션

턴 수에 따른 **총 입력 토큰 합계**가 선형이 아닌 **이차(quadratic)** 성장을
보이는 이유를 수학적으로 확인하고 시각화합니다.

In [ ]:
import numpy as np

def total_input_tokens_formula(n: int, base: int = 1000, avg_output: int = 800) -> int:
    """
    n턴 에이전트의 총 입력 토큰 합계를 수학적으로 계산합니다.
    
    Turn k의 입력 = base + avg_output * (k - 1)
    총합 = n * base + avg_output * n * (n - 1) / 2
    → O(n²) 성장
    """
    return int(n * base + avg_output * n * (n - 1) / 2)

# 1~30턴까지 총 입력 토큰 계산
turns_range = range(1, 31)
total_inputs = [total_input_tokens_formula(n) for n in turns_range]

# 선형 성장 비교 (만약 컨텍스트 누적이 없다면)
linear_inputs = [n * (1000 + 800) for n in turns_range]  # base + avg_output per turn

# 비교 테이블
comparison_turns = [1, 5, 10, 15, 20, 25, 30]
print("📊 컨텍스트 누적 vs 비누적 총 입력 토큰")
print(f"{'턴':>5} {'누적 (실제)':>15} {'비누적 (가정)':>15} {'배율':>8}")
print("-" * 48)
for n in comparison_turns:
    actual = total_input_tokens_formula(n)
    linear = n * (1000 + 800)
    ratio = actual / linear
    print(f"{n:>5} {actual:>15,} {linear:>15,} {ratio:>8.2f}x")

print("\n💡 30턴에서 컨텍스트 누적 시 비누적 대비 7.4x 더 많은 입력 토큰!")
print("완료 ✓")

In [ ]:
# O(n²) vs O(n) 성장 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

turns = list(turns_range)

# 1) 총 입력 토큰: 누적 vs 비누적
axes[0].plot(turns, total_inputs, marker="o", linewidth=2, color="#EF5350",
             label="컨텍스트 누적 (O(n²))", markersize=4)
axes[0].plot(turns, linear_inputs, marker="s", linewidth=2, color="#42A5F5",
             label="비누적 (O(n))", markersize=4)
axes[0].set_xlabel("턴 수")
axes[0].set_ylabel("총 입력 토큰")
axes[0].set_title("컨텍스트 누적: O(n²) vs O(n) 성장")
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# 2) 비용 차이 (Claude Sonnet 4 기준)
input_price = 3.0  # $/M tokens
costs_accumulated = [t * input_price / 1_000_000 for t in total_inputs]
costs_linear = [t * input_price / 1_000_000 for t in linear_inputs]

axes[1].fill_between(turns, costs_accumulated, costs_linear, alpha=0.3, color="#EF5350",
                     label="추가 비용 (누적으로 인한)")
axes[1].plot(turns, costs_accumulated, linewidth=2, color="#EF5350", label="누적 비용")
axes[1].plot(turns, costs_linear, linewidth=2, color="#42A5F5", label="비누적 비용")
axes[1].set_xlabel("턴 수")
axes[1].set_ylabel("입력 토큰 비용 ($)")
axes[1].set_title("입력 토큰 비용: 누적 vs 비누적 (Sonnet 4)")
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("완료 ✓")

---
## 섹션 7. 모델별 비용 비교 DataFrame

2026년 기준 주요 LLM 모델의 가격, 컨텍스트 윈도우, 속도를 정리합니다.

In [ ]:
# 2026년 기준 주요 모델 비용 비교표
model_data = [
    {"모델": "Claude Opus 4", "제공사": "Anthropic", "입력 ($/M)": 15.00,
     "출력 ($/M)": 75.00, "컨텍스트": "200K", "속도": "느림", "추천 용도": "고난이도 추론"},
    {"모델": "Claude Sonnet 4", "제공사": "Anthropic", "입력 ($/M)": 3.00,
     "출력 ($/M)": 15.00, "컨텍스트": "200K", "속도": "빠름", "추천 용도": "코드 생성, 분석"},
    {"모델": "Claude Haiku 3.5", "제공사": "Anthropic", "입력 ($/M)": 0.80,
     "출력 ($/M)": 4.00, "컨텍스트": "200K", "속도": "매우빠름", "추천 용도": "분류, 추출"},
    {"모델": "GPT-4o", "제공사": "OpenAI", "입력 ($/M)": 2.50,
     "출력 ($/M)": 10.00, "컨텍스트": "128K", "속도": "빠름", "추천 용도": "범용"},
    {"모델": "GPT-4o mini", "제공사": "OpenAI", "입력 ($/M)": 0.15,
     "출력 ($/M)": 0.60, "컨텍스트": "128K", "속도": "매우빠름", "추천 용도": "경량 작업"},
    {"모델": "Gemini 2.5 Pro", "제공사": "Google", "입력 ($/M)": 1.25,
     "출력 ($/M)": 10.00, "컨텍스트": "1M", "속도": "보통", "추천 용도": "장문 분석"},
    {"모델": "Gemini 2.5 Flash", "제공사": "Google", "입력 ($/M)": 0.15,
     "출력 ($/M)": 0.60, "컨텍스트": "1M", "속도": "매우빠름", "추천 용도": "경량 + 장문"},
    {"모델": "Llama 3.3 70B", "제공사": "Meta (자체호스팅)", "입력 ($/M)": 0.40,
     "출력 ($/M)": 0.40, "컨텍스트": "128K", "속도": "빠름", "추천 용도": "자체 호스팅"},
]

df_models = pd.DataFrame(model_data)

# 출력/입력 가격 배율 계산
df_models["출력/입력 배율"] = (df_models["출력 ($/M)"] / df_models["입력 ($/M)"]).round(1)

print("📊 2026년 주요 LLM 모델 비용 비교표")
print(df_models.to_string(index=False))
print("완료 ✓")

In [ ]:
# 모델별 비용 시각화: 10턴 에이전트 비용 시뮬레이션
agent_costs = []
for _, row in df_models.iterrows():
    total_in = total_input_tokens_formula(10, base=1000, avg_output=800)
    total_out = 800 * 10  # 평균 출력
    cost = calc_cost(total_in, total_out, row["입력 ($/M)"], row["출력 ($/M)"])
    agent_costs.append({"모델": row["모델"], "10턴 비용 ($)": round(cost, 4)})

df_agent_costs = pd.DataFrame(agent_costs).sort_values("10턴 비용 ($)", ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#4CAF50" if c < 0.05 else "#FF9800" if c < 0.2 else "#F44336"
          for c in df_agent_costs["10턴 비용 ($)"]]
ax.barh(df_agent_costs["모델"], df_agent_costs["10턴 비용 ($)"], color=colors)
ax.set_xlabel("10턴 에이전트 비용 ($)")
ax.set_title("모델별 10턴 에이전트 예상 비용")

# 값 표시
for i, (_, row) in enumerate(df_agent_costs.iterrows()):
    ax.text(row["10턴 비용 ($)"] + 0.005, i, f"${row['10턴 비용 ($)']:.4f}",
            va="center", fontsize=10)

plt.tight_layout()
plt.show()
print("완료 ✓")

---
## 섹션 8. Langfuse 비용 태깅 구현

Langfuse `CallbackHandler`를 사용하여 LLM 호출에 비용 태그를 부착하고,
모델/기능/팀별로 비용을 추적하는 방법을 구현합니다.

In [ ]:
from langchain_anthropic import ChatAnthropic
from langfuse.callback import CallbackHandler

# Langfuse CallbackHandler 생성 함수
def create_langfuse_handler(feature: str, team: str, env: str = "dev",
                             session_id: str = None) -> CallbackHandler:
    """
    태그가 포함된 Langfuse CallbackHandler를 생성합니다.
    
    Args:
        feature: 기능 태그 (예: code-review, summarize)
        team: 팀 태그 (예: backend, data)
        env: 환경 태그 (예: dev, staging, prod)
        session_id: 세션 ID (선택)
    """
    handler = CallbackHandler(
        public_key=os.environ.get("LANGFUSE_PUBLIC_KEY", ""),
        secret_key=os.environ.get("LANGFUSE_SECRET_KEY", ""),
        host="https://cloud.langfuse.com",
        tags=[f"feature:{feature}", f"team:{team}", f"env:{env}"],
        session_id=session_id or f"session_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
        metadata={
            "feature": feature,
            "team": team,
            "env": env,
            "timestamp": datetime.now().isoformat(),
        }
    )
    return handler

print("Langfuse CallbackHandler 생성 함수 정의 완료 ✓")
print("\n사용 예시:")
print('  handler = create_langfuse_handler(feature="code-review", team="backend")')
print('  llm.invoke("질문", config={"callbacks": [handler]})')

In [ ]:
# Langfuse 태깅이 포함된 LLM 호출 실행
# (Langfuse 키가 설정되지 않으면 태깅 없이 실행)

llm = ChatAnthropic(
    model="claude-sonnet-4-20250514",
    temperature=0,
    max_tokens=500,
)

# 태그가 포함된 CallbackHandler 생성
handler = create_langfuse_handler(
    feature="token-economics-demo",
    team="education",
    env="dev"
)

# LLM 호출 (비용 추적 포함)
response = llm.invoke(
    "LLM 비용 최적화에서 가장 중요한 3가지 전략을 간단히 설명해줘.",
    config={"callbacks": [handler]}
)

print("📤 LLM 응답:")
print(response.content)
print("\n📊 사용량 메타데이터:")
if hasattr(response, 'usage_metadata') and response.usage_metadata:
    print(f"  입력 토큰: {response.usage_metadata.get('input_tokens', 'N/A')}")
    print(f"  출력 토큰: {response.usage_metadata.get('output_tokens', 'N/A')}")
print("\n→ Langfuse 대시보드에서 태그별 비용을 확인할 수 있습니다.")
print("완료 ✓")

---
## 섹션 9. 비용 대시보드 데이터 집계

실제 Langfuse 대시보드를 구축하기 전에,
비용 데이터를 **모델별, 기능별, 시간대별**로 집계하는 방법을 구현합니다.

In [ ]:
# 시뮬레이션 비용 데이터 생성 (실제 환경에서는 Langfuse API로 조회)
import random
random.seed(42)

# 7일간의 비용 데이터 시뮬레이션
features = ["code-review", "summarize", "translate", "qa-bot", "data-analysis"]
teams = ["backend", "frontend", "data", "infra"]
model_names = ["claude-sonnet-4", "claude-haiku-3.5", "gpt-4o-mini"]
model_prices = {
    "claude-sonnet-4": {"input": 3.0, "output": 15.0},
    "claude-haiku-3.5": {"input": 0.8, "output": 4.0},
    "gpt-4o-mini": {"input": 0.15, "output": 0.6},
}

cost_records = []
for day_offset in range(7):
    date = datetime(2026, 4, 1) + timedelta(days=day_offset)
    n_calls = random.randint(50, 200)
    
    for _ in range(n_calls):
        model = random.choice(model_names)
        feature = random.choice(features)
        team = random.choice(teams)
        input_tokens = random.randint(500, 10000)
        output_tokens = random.randint(200, 3000)
        prices = model_prices[model]
        cost = calc_cost(input_tokens, output_tokens, prices["input"], prices["output"])
        
        cost_records.append({
            "날짜": date.strftime("%Y-%m-%d"),
            "모델": model,
            "기능": feature,
            "팀": team,
            "입력 토큰": input_tokens,
            "출력 토큰": output_tokens,
            "비용 ($)": round(cost, 6),
        })

df_costs = pd.DataFrame(cost_records)
print(f"📊 시뮬레이션 데이터: {len(df_costs)}건의 호출 기록")
print(f"   기간: {df_costs['날짜'].min()} ~ {df_costs['날짜'].max()}")
print(f"   총 비용: ${df_costs['비용 ($)'].sum():.2f}")
print(df_costs.head(10).to_string(index=False))
print("완료 ✓")

In [ ]:
# 비용 대시보드: 3가지 관점으로 집계
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1) 일별 비용 추이
daily_cost = df_costs.groupby("날짜")["비용 ($)"].sum()
axes[0, 0].bar(range(len(daily_cost)), daily_cost.values, color="#42A5F5")
axes[0, 0].set_xticks(range(len(daily_cost)))
axes[0, 0].set_xticklabels(daily_cost.index, rotation=45, ha="right")
axes[0, 0].set_ylabel("비용 ($)")
axes[0, 0].set_title("일별 총 비용 추이")
axes[0, 0].axhline(y=daily_cost.mean(), color="red", linestyle="--", label=f"평균: ${daily_cost.mean():.2f}")
axes[0, 0].legend()

# 2) 모델별 비용 비율
model_cost = df_costs.groupby("모델")["비용 ($)"].sum().sort_values(ascending=False)
axes[0, 1].pie(model_cost.values, labels=model_cost.index, autopct="%1.1f%%",
               colors=["#EF5350", "#42A5F5", "#66BB6A"], startangle=90)
axes[0, 1].set_title("모델별 비용 비율")

# 3) 기능별 비용 TOP 5
feature_cost = df_costs.groupby("기능")["비용 ($)"].sum().sort_values(ascending=True)
axes[1, 0].barh(feature_cost.index, feature_cost.values, color="#FF7043")
axes[1, 0].set_xlabel("비용 ($)")
axes[1, 0].set_title("기능별 비용")

# 4) 팀별 비용
team_cost = df_costs.groupby("팀")["비용 ($)"].sum().sort_values(ascending=True)
axes[1, 1].barh(team_cost.index, team_cost.values, color="#AB47BC")
axes[1, 1].set_xlabel("비용 ($)")
axes[1, 1].set_title("팀별 비용")

plt.suptitle("LLM 비용 대시보드 (7일 시뮬레이션)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()
print("완료 ✓")

---
## 섹션 10. 비용 알림 시스템

일일 비용이 임계값을 초과하면 자동으로 알림을 발생시키는
**threshold-based alert** 시스템을 구현합니다.

In [ ]:
# 비용 알림 시스템 구현

class AlertLevel(Enum):
    """알림 수준 정의"""
    NORMAL = "normal"
    WARNING = "warning"
    CRITICAL = "critical"
    EMERGENCY = "emergency"

@dataclass
class CostAlert:
    """비용 알림 데이터 클래스"""
    level: AlertLevel
    daily_cost: float
    threshold: float
    message: str
    actions: list

class CostAlertSystem:
    """비용 알림 시스템"""
    
    def __init__(self, thresholds: dict = None):
        # 기본 임계값 (일일 비용 기준, 단위: $)
        self.thresholds = thresholds or {
            "warning": 50.0,
            "critical": 100.0,
            "emergency": 200.0,
        }
        self.alert_history = []
    
    def check(self, daily_cost: float) -> CostAlert:
        """일일 비용을 확인하고 알림 레벨을 결정합니다."""
        if daily_cost >= self.thresholds["emergency"]:
            alert = CostAlert(
                level=AlertLevel.EMERGENCY,
                daily_cost=daily_cost,
                threshold=self.thresholds["emergency"],
                message=f"비상! 일일 비용 ${daily_cost:.2f} > ${self.thresholds['emergency']:.2f}",
                actions=["자동 트래픽 제한 발동", "경영진 보고", "Slack + Email 알림"]
            )
        elif daily_cost >= self.thresholds["critical"]:
            alert = CostAlert(
                level=AlertLevel.CRITICAL,
                daily_cost=daily_cost,
                threshold=self.thresholds["critical"],
                message=f"위험: 일일 비용 ${daily_cost:.2f} > ${self.thresholds['critical']:.2f}",
                actions=["Slack + Email 알림", "비용 원인 분석 시작"]
            )
        elif daily_cost >= self.thresholds["warning"]:
            alert = CostAlert(
                level=AlertLevel.WARNING,
                daily_cost=daily_cost,
                threshold=self.thresholds["warning"],
                message=f"경고: 일일 비용 ${daily_cost:.2f} > ${self.thresholds['warning']:.2f}",
                actions=["Slack 알림 전송", "팀 리더에게 통보"]
            )
        else:
            alert = CostAlert(
                level=AlertLevel.NORMAL,
                daily_cost=daily_cost,
                threshold=self.thresholds["warning"],
                message=f"정상: 일일 비용 ${daily_cost:.2f}",
                actions=["모니터링 계속"]
            )
        
        self.alert_history.append(alert)
        return alert
    
    def display_alert(self, alert: CostAlert):
        """알림을 시각적으로 출력합니다."""
        level_icons = {
            AlertLevel.NORMAL: "✅",
            AlertLevel.WARNING: "⚠️",
            AlertLevel.CRITICAL: "🔴",
            AlertLevel.EMERGENCY: "🚨",
        }
        icon = level_icons[alert.level]
        print(f"\n{icon} [{alert.level.value.upper()}] {alert.message}")
        print(f"   액션:")
        for action in alert.actions:
            print(f"   - {action}")

print("비용 알림 시스템 정의 완료 ✓")

In [ ]:
# 알림 시스템 테스트: 시뮬레이션 데이터로 검증
alert_system = CostAlertSystem(thresholds={
    "warning": 50.0,
    "critical": 100.0,
    "emergency": 200.0,
})

# 다양한 비용 시나리오 테스트
test_costs = [25.0, 55.0, 120.0, 250.0, 30.0]

print("📊 비용 알림 시스템 테스트")
print("=" * 50)

for cost in test_costs:
    alert = alert_system.check(cost)
    alert_system.display_alert(alert)

# 실제 시뮬레이션 데이터에 적용
print("\n" + "=" * 50)
print("\n📊 7일간 시뮬레이션 데이터에 알림 적용")
daily_costs = df_costs.groupby("날짜")["비용 ($)"].sum()

# 시뮬레이션 비용을 확대 (데모 목적: 실제 비용 x 1000)
for date, cost in daily_costs.items():
    scaled_cost = cost * 1000  # 데모용 스케일링
    alert = alert_system.check(scaled_cost)
    print(f"\n  {date}: ${scaled_cost:.2f} → [{alert.level.value.upper()}]")

print("\n완료 ✓")

---
## 섹션 11. 비용 최적화 전략 정리

지금까지 배운 내용을 바탕으로 **비용 최적화 전략**을 체계적으로 정리합니다.

In [ ]:
# 비용 최적화 전략 정리 테이블
strategies = [
    {
        "전략": "프롬프트 최적화",
        "설명": "시스템 프롬프트 간결화, Few-shot 최소화, 불필요한 지시문 제거",
        "구현 난이도": "낮음",
        "예상 절감": "10~30%",
        "적용 시점": "항상 먼저 적용",
    },
    {
        "전략": "출력 길이 제한",
        "설명": "max_tokens 설정, 출력 포맷 제한 (JSON, 구조화)",
        "구현 난이도": "낮음",
        "예상 절감": "20~40%",
        "적용 시점": "출력이 불필요하게 길 때",
    },
    {
        "전략": "Model Routing",
        "설명": "작업 복잡도에 따라 Haiku/Sonnet/Opus 자동 분배",
        "구현 난이도": "중간",
        "예상 절감": "60~80%",
        "적용 시점": "즉시 적용 가능",
    },
    {
        "전략": "Context Compression",
        "설명": "긴 컨텍스트를 요약 후 재주입, 불필요한 이전 턴 제거",
        "구현 난이도": "중간",
        "예상 절감": "30~50%",
        "적용 시점": "장문 에이전트에 효과적",
    },
    {
        "전략": "Semantic Cache",
        "설명": "동일/유사 질문에 캐시된 응답 반환, 임베딩 유사도 기반",
        "구현 난이도": "높음",
        "예상 절감": "최대 90%",
        "적용 시점": "반복 패턴이 있을 때",
    },
    {
        "전략": "배치 API",
        "설명": "비실시간 작업을 배치로 묶어 50% 할인 가격 적용",
        "구현 난이도": "낮음",
        "예상 절감": "50%",
        "적용 시점": "비실시간 작업 (보고서, 분석)",
    },
]

df_strategies = pd.DataFrame(strategies)

print("📊 LLM 비용 최적화 전략 정리")
print("=" * 90)
print(df_strategies.to_string(index=False))

print("\n🏆 ROI 순위: 프롬프트 최적화 > Model Routing > 출력 제한 > 배치 API > 캐싱 > 압축")
print("완료 ✓")

In [ ]:
# Model Routing 시뮬레이션: 전략 적용 전/후 비용 비교

# 현재: 모든 작업에 Sonnet 4 사용
# 최적화: 복잡도에 따라 Haiku/Sonnet/Opus 분배

# 작업 분포 가정
task_distribution = {
    "단순 (분류, 추출)": {"비율": 0.60, "tokens": 2000, "before": "sonnet-4", "after": "haiku-3.5"},
    "중간 (요약, 분석)": {"비율": 0.30, "tokens": 5000, "before": "sonnet-4", "after": "sonnet-4"},
    "고난이도 (추론)": {"비율": 0.10, "tokens": 10000, "before": "sonnet-4", "after": "opus-4"},
}

model_prices_routing = {
    "haiku-3.5": {"input": 0.8, "output": 4.0},
    "sonnet-4": {"input": 3.0, "output": 15.0},
    "opus-4": {"input": 15.0, "output": 75.0},
}

daily_calls = 1000
before_cost = 0
after_cost = 0

print("📊 Model Routing 비용 시뮬레이션 (일일 1,000 호출)")
print(f"{'작업 유형':<20} {'비율':>6} {'Before ($)':>12} {'After ($)':>12} {'절감 ($)':>12}")
print("-" * 65)

for task_name, task in task_distribution.items():
    n_calls = int(daily_calls * task["비율"])
    tokens = task["tokens"]
    
    # Before: 모두 Sonnet
    p_before = model_prices_routing[task["before"]]
    cost_before = n_calls * calc_cost(tokens, tokens // 2, p_before["input"], p_before["output"])
    
    # After: 라우팅 적용
    p_after = model_prices_routing[task["after"]]
    cost_after = n_calls * calc_cost(tokens, tokens // 2, p_after["input"], p_after["output"])
    
    saving = cost_before - cost_after
    before_cost += cost_before
    after_cost += cost_after
    
    print(f"{task_name:<20} {task['비율']:>5.0%} ${cost_before:>11.2f} ${cost_after:>11.2f} ${saving:>11.2f}")

print("-" * 65)
print(f"{'합계':<20} {'100%':>6} ${before_cost:>11.2f} ${after_cost:>11.2f} ${before_cost - after_cost:>11.2f}")
print(f"\n💰 절감률: {(1 - after_cost / before_cost) * 100:.1f}%")
print(f"💰 월간 절감 (30일): ${(before_cost - after_cost) * 30:.2f}")
print("완료 ✓")

---
## 섹션 12. Exercise

아래 두 가지 실습을 완성해보세요.

### Exercise 1: 커스텀 비용 시뮬레이터

**목표**: 다양한 모델과 턴 수에 따른 비용을 비교하는 시뮬레이터를 구현하세요.

**요구사항**:
1. 3개 이상의 모델을 동시에 시뮬레이션
2. 턴 수를 1~20까지 변화시키며 비용 계산
3. matplotlib로 모델별 비용 비교 라인 차트 생성
4. 가장 비용 효율적인 모델과 턴 수를 출력

In [ ]:
# === Exercise 1: 커스텀 비용 시뮬레이터 ===
# TODO: 아래 코드를 완성하세요

def compare_model_costs(models_config: dict, max_turns: int = 20,
                         base_tokens: int = 1000, avg_output: int = 800):
    """
    여러 모델의 턴별 비용을 비교합니다.
    
    Args:
        models_config: {"모델명": {"input": 가격, "output": 가격}}
        max_turns: 최대 턴 수
        base_tokens: 기본 입력 토큰
        avg_output: 평균 출력 토큰
    
    Returns:
        DataFrame with costs per model per turn
    """
    # TODO: 구현하세요
    # 힌트: simulate_agent_turns() 함수를 참고하세요
    # 1. 각 모델에 대해 1~max_turns까지 비용 계산
    # 2. 결과를 DataFrame으로 반환
    pass

# TODO: 시뮬레이션 실행
# models_config = {
#     "Claude Sonnet 4": {"input": 3.0, "output": 15.0},
#     "Claude Haiku 3.5": {"input": 0.8, "output": 4.0},
#     "GPT-4o mini": {"input": 0.15, "output": 0.6},
# }
# df_comparison = compare_model_costs(models_config)

# TODO: matplotlib로 비교 차트 생성

# TODO: 가장 비용 효율적인 모델 출력

print("Exercise 1 - TODO: 위 코드를 완성하세요")

### Exercise 2: 비용 알림 대시보드 확장

**목표**: 비용 알림 시스템에 **팀별/기능별 세분화 알림**을 추가하세요.

**요구사항**:
1. 팀별 일일 비용 임계값 설정 (팀마다 다른 예산)
2. 기능별 비용이 특정 비율을 초과하면 알림
3. 비용 추세가 전일 대비 50% 이상 증가하면 알림
4. 알림 히스토리를 DataFrame으로 출력

In [ ]:
# === Exercise 2: 비용 알림 대시보드 확장 ===
# TODO: 아래 코드를 완성하세요

class AdvancedCostAlertSystem:
    """
    팀별/기능별 세분화 알림을 지원하는 고급 비용 알림 시스템
    """
    
    def __init__(self, team_budgets: dict, feature_limits: dict):
        """
        Args:
            team_budgets: {"팀명": 일일_예산($)} 예: {"backend": 80, "data": 120}
            feature_limits: {"기능명": 비용_비율_한도} 예: {"code-review": 0.4}
        """
        # TODO: 초기화
        self.team_budgets = team_budgets
        self.feature_limits = feature_limits
        self.alert_history = []
    
    def check_team_budget(self, team: str, daily_cost: float):
        """팀별 예산 초과 여부를 확인합니다."""
        # TODO: 구현하세요
        pass
    
    def check_feature_ratio(self, feature: str, feature_cost: float, total_cost: float):
        """기능별 비용 비율 초과 여부를 확인합니다."""
        # TODO: 구현하세요
        pass
    
    def check_trend(self, today_cost: float, yesterday_cost: float):
        """전일 대비 비용 증가 추세를 확인합니다."""
        # TODO: 구현하세요
        pass
    
    def get_alert_report(self) -> pd.DataFrame:
        """알림 히스토리를 DataFrame으로 반환합니다."""
        # TODO: 구현하세요
        pass

# TODO: 시뮬레이션 데이터(df_costs)로 테스트

print("Exercise 2 - TODO: 위 코드를 완성하세요")

---
## 마무리

### 오늘 배운 것

1. **에이전트 비용 구조**: Multi-turn Loop에서 컨텍스트 누적으로 챗봇 대비 3~10x 비용 발생
2. **O(n²) 성장**: 턴 수가 늘수록 총 입력 토큰이 이차 함수적으로 증가
3. **입출력 비대칭**: 출력 토큰이 입력보다 4~8배 비싸서 전체 비용의 60%+ 차지
4. **비용 추적**: Langfuse 태깅으로 모델/기능/팀별 비용을 실시간 추적
5. **알림 시스템**: 임계값 기반 Normal/Warning/Critical/Emergency 4단계 알림
6. **최적화 전략**: Model Routing(60~80%), Context Compression(30~50%), Semantic Cache(~90%)

### 다음 에피소드

**EP16**: Model Routing 심화 -- 작업별 최적 모델을 자동으로 선택하는 라우팅 파이프라인 구축

> 전체 코드는 GitHub 레포에서, 심화 내용은 커뮤니티에서